In [1]:
import torch
import torch.nn as nn
from ultralytics import YOLO

model = YOLO("best.pt")
print(model.model)

class TopKPoolWithCoords(nn.Module):
    def __init__(self, k=5):
        super().__init__()
        self.k = k

    def forward(self, x):
        B, C, H, W = x.shape

        flat = x.reshape(B, C, -1)

        vals, idxs = flat.topk(self.k, dim=2)

        # Coordinates
        y = (idxs // W).float() / H
        x_coord = (idxs % W).float() / W

        result = torch.stack([vals, x_coord, y], dim=3)
        # Flatten last two dims to get (B, C, 3k)
        result = result.reshape(B, C, -1)

        return result # (B, C, 3k)



class ViolenceYOLO(nn.Module):
    def __init__(self, p3_layer=16, p4_layer=19, p5_layer=22):
        super().__init__()
        yolo = YOLO("best.pt")
        self.feat_layer1 = p3_layer
        self.feat_layer2 = p4_layer
        self.feat_layer3 = p5_layer

        self.model = yolo.model
        self.model.model[-1].max_det = 5
        del yolo

        self.pool = TopKPoolWithCoords(k=5)

    def forward(self, x):
        y = []

        p3 = p4 = p5 = None

        for i, layer in enumerate(self.model.model):

            if hasattr(layer, "f"):
                if isinstance(layer.f, int):
                    x_in = y[layer.f] if layer.f != -1 else x
                else:
                    x_in = [y[j] for j in layer.f]
            else:
                x_in = x

            x = layer(x_in)

            y.append(x)

            if i == self.feat_layer1:
                p3 = self.pool(x)
            elif i == self.feat_layer2:
                p4 = self.pool(x)
            elif i == self.feat_layer3:
                p5 = self.pool(x)

        feats = torch.cat([p3, p4, p5], dim=1) # (B, C3, 3k)
        preds = x[0]
        return preds, feats


DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
 

In [2]:
# 1. Initialize your custom model
device = torch.device("cpu") # CPU is usually safer for the export process
model = ViolenceYOLO()
print(type(model))
model.eval()

# 2. Create dummy input (Standard YOLO input: Batch, Channels, Height, Width)
# Use the same resolution you plan to use in production (320p)
dummy_input = torch.randn(1, 3, 320, 256).to(device)

results = model(dummy_input)

print("YOLO output shape:", results[0].shape) # (B, num_predictions, 85) ex: (1, 15, 85)
print("Feature output shape:", results[1].shape) # (B, C1+C2+C3, 3k) ex: (1, 512+256+128, 15) = (1, 896, 15)

<class '__main__.ViolenceYOLO'>
YOLO output shape: torch.Size([1, 5, 6])
Feature output shape: torch.Size([1, 896, 15])


In [3]:
# 3. Export using torch.onnx
torch.onnx.export(
    model,
    dummy_input,
    "violence_yolo.onnx",
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=['images'],
    output_names=['output0', 'features'],
    dynamic_axes={
        'images': {0: 'batch_size'},
        'output0': {0: 'batch_size'},
        'features': {0: 'batch_size'}
    },
    dynamo=False   # critical
)


C:\Users\Dell\AppData\Local\Temp\ipykernel_13520\2046344125.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 13 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


In [4]:
import onnxruntime as ort
import numpy as np
import cv2
import time

# ---- Load model ----
session = ort.InferenceSession(
    "violence_yolo.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name

# ---- Prepare image ----
img = cv2.imread("demo/img1.png")
img = cv2.resize(img, (256, 320))
img = img.transpose(2, 0, 1)
img = np.expand_dims(img, axis=0).astype(np.float32) / 255.0

# ---- Warmup (very important) ----
for _ in range(10):
    session.run(None, {input_name: img})

# ---- Speed Test ----
runs = 100
start = time.time()

for _ in range(runs):
    outputs = session.run(None, {input_name: img})
   

end = time.time()
print(f"Output shapes: {[o.shape for o in outputs]}")
avg_time = (end - start) / runs
fps = 1 / avg_time

print(f"Average inference time: {avg_time*1000:.2f} ms")
print(f"FPS: {fps:.2f}")



Output shapes: [(1, 5, 6), (1, 896, 15)]
Average inference time: 32.59 ms
FPS: 30.69


In [ ]:
import cv2
import numpy as np
import onnxruntime as ort
import time

# Load ONNX model
session = ort.InferenceSession("violence_yolo.onnx", providers=["CPUExecutionProvider"])

input_name = session.get_inputs()[0].name

cap = cv2.VideoCapture("demovid/vid6.avi")  # or path to video

while True:
    ret, frame = cap.read()
    if not ret:
        break

    start = time.time()

    # Resize to model size (example: 320x320)
    img = cv2.resize(frame, (320, 320))
    img = img[:, :, ::-1]  # BGR → RGB
    img = img.transpose(2, 0, 1)  # HWC → CHW
    img = np.expand_dims(img, axis=0).astype(np.float32) / 255.0

    outputs = session.run(None, {input_name: img})

    detections = outputs[0]      # (1, 5, 6)
    feature    = outputs[-1]     # (1, 512)

    # Draw detections
    for det in detections[0]:
        x1, y1, x2, y2, conf, cls = det

        if conf > 0.1:
            x1 = int(x1 * frame.shape[1] / 320)
            y1 = int(y1 * frame.shape[0] / 320)
            x2 = int(x2 * frame.shape[1] / 320)
            y2 = int(y2 * frame.shape[0] / 320)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, f"{conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    fps = 1 / (time.time() - start)
    cv2.putText(frame, f"FPS: {fps:.2f}", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("Demo", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
model = YOLO("best.pt")
model.export(format="onnx", dynamic=True)

In [ ]:
import onnxruntime as ort
import numpy as np
import cv2
import time

# ---- Load model ----
session = ort.InferenceSession(
    "best.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name

# ---- Prepare image ----
img = cv2.imread("demo/img1.png")
img = cv2.resize(img, (256, 320))
img = img.transpose(2, 0, 1)
img = np.expand_dims(img, axis=0).astype(np.float32) / 255.0

# ---- Warmup (very important) ----
for _ in range(10):
    session.run(None, {input_name: img})

# ---- Speed Test ----
runs = 100
start = time.time()

for _ in range(runs):
    outputs = session.run(None, {input_name: img})
   

end = time.time()
print(f"Output shapes: {[o.shape for o in outputs]}")
avg_time = (end - start) / runs
fps = 1 / avg_time

print(f"Average inference time: {avg_time*1000:.2f} ms")
print(f"FPS: {fps:.2f}")
